In [1]:
DATA_SOURCES = ["tag"]

# Setup

## Processor-Specific

In [2]:
%load_ext autoreload
%autoreload 2
import os
import sys
from dotenv import load_dotenv
from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("..")
load_dotenv("../../.env")

from pneuma_seeker.core.conductor.chat_interface import ChatInterface
from pneuma_seeker.model.interface.impl.gpt import GPT
from pneuma_seeker.core.ir_system.data_model import convert_multi_retriever_results_to_str

In [3]:
llm_path = "o4-mini"
embed_model_path = "model/weight/bge-base"
chat_interface = ChatInterface(llm_path, embed_model_path, "user_id", "chat_id", DATA_SOURCES)
gpt = GPT("gpt-4o")

[2025-08-29 11:43:52] INFO in main: Initializing Materializer


## Benchmark-Specific

In [4]:
import pandas as pd

benchmark = pd.read_csv("../../benchmark/sources/TAG/tag_queries.csv")

In [5]:
def get_formatted_system_output(ci_output):
    system_output = ci_output['system_response']
    state = ci_output['state']
    current_retrieval_results = ci_output['current_retrieval_results']

    return f"""SYSTEM OUTPUT:
```{system_output}```

STATE:
```{state}```

RETRIEVED DATA BY THE SYSTEM:
```{convert_multi_retriever_results_to_str(current_retrieval_results)}```
"""

In [6]:
def get_prompt_to_llm(question: str) -> str:
    """
    Returns a goal-focused system prompt for an LLM that will interact with
    a data-assistant system to produce a specific answer efficiently.
    """
    return f"""You are an expert analyst using a data assistant system to answer a specific question.
The system expresses its understanding as:
- a set of target schemas (tables it thinks are relevant)
- a list of SQL statements which, if run sequentially on these schemas, should produce an answer.

Your job is to:
1. Evaluate whether the system's representation (schemas + SQL) correctly matches the question.
2. If it does, confirm and proceed.
3. If it doesn't, refine your instructions or clarify requirements so the system aligns with your goal.

**Important:** Your ultimate objective is to correctly and efficiently answer this question:
```

{question}

```

Guidelines for your responses:
- Act like a domain expert (precise, critical, but not verbose).
- If the system misinterprets your need, correct it directly.
- Keep focus on the main question — only explore side ideas if they help clarify or validate the path to the answer.
- Speak to the system as if you are giving it instructions or feedback — do not roleplay with humans or produce narrative explanations.

Continue the conversation from here, giving your next message to the system:

YOU: {question}
""".strip()

# INTERACTION

In [7]:
import time
from pneuma_seeker.model.llm_message import LLMMessage, Role
from pneuma_seeker.model.option import LLMOption

In [8]:
index = 0
iteration = -1
QUESTION = benchmark['Query'][0]
ITERATION_LIMIT = 15
print(f"Question: {QUESTION}")

Question: Among the schools with the average score in Math over 560 in the SAT test, how many schools are in counties in the bay area?


In [9]:
gpt_init_prompt = get_prompt_to_llm(QUESTION)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = QUESTION
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

=> CURRENT USER PROMPT: Among the schools with the average score in Math over 560 in the SAT test, how many schools are in counties in the bay area?


In [ ]:
iteration += 1
start = time.time()
system_output = chat_interface.process_user_input(curr_user_prompt)
end = time.time()
print(f"===> SYSTEM OUTPUT: {system_output}")
format_to_gpt = get_formatted_system_output(system_output)
if iteration == 0:
    gpt_messages[0]['content'] += f"\n{format_to_gpt}"
else:
    gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
if updated_user_prompt.startswith("YOU:"):
    updated_user_prompt = updated_user_prompt[4:]
    updated_user_prompt = updated_user_prompt.strip()
curr_user_prompt = updated_user_prompt
print(f"Responding time: {end-start} seconds")
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

===> SYSTEM OUTPUT: <generator object ChatInterface.process_user_input at 0x7f40f46d3560>


TypeError: 'generator' object is not subscriptable

# FINAL

In [ ]:
import json

def write_jsonl(data, file_path):
    """Writes a list of JSON objects (dicts) to a JSONL file."""
    with open(file_path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

write_jsonl(f"benchmark_data/O4_{DATA_SOURCES}_{index+1}.jsonl", gpt_messages, True)